# Behavioral Data Simulation: Credit Card Fraud
By: Jared Alviar

---

###Projected Goal
The objective of this project is to simulate an enterprise financial transaction environment and evaluate an automated fraud detection pipeline under realistic class imbalance. Using NumPy and Pandas, we synthesize 100,000 credit card transactions mapped across 10,000 unique users, inject behavioral anomaly signatures reflecting real-world fraud schemes, and evaluate a baseline classifier on a held-out test split.

### Key Features
- **Scalable Data Simulation:** Generates 100,000 raw transaction records cross-referenced with 10,000 user profiles, maintaining a realistic 5% fraud class distribution (95% legitimate, 5% fraudulent).
- **Behavioral Pattern Injection:** Simulates three distinct fraudulent mechanisms:
  - *Pattern A (Financial Profile Mismatch):* Low credit scores (300–500) paired with elevated annual incomes (120k–500k), reflecting compromised high-value accounts.
  - *Pattern B (Temporal Off-Hours Exploitation):* Skewed transaction activity concentrated between 12:00 AM and 5:00 AM to bypass immediate cardholder alerts.
  - *Pattern C (Merchant Category Bias):* High-liquidity merchant targeting focused exclusively in Electronics and Travel.
- **Empirical Held-Out Validation:** Evaluates an 80/20 stratified Random Forest baseline (`n_estimators=100`, `max_depth=10`) on 20,000 unseen test records, achieving **1.0000 Precision**, **0.7980 Recall**, and **0.9986 ROC-AUC**.



In [2]:
# ENVIRONMENT SETUP
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Setting a seed for reproducibility
np.random.seed(7)

## Step 1: Data Generation
In this phase, we will create our synthetic data set. We are starting with 10000 users and making them have 10 transactions each (100,000 transactions each).

In [3]:
# DATA INITIALIZATION

n = 10000

# The rules of my data set
data = {
    "user_id": range(1, n+1),
    "income": np.random.randint(10000, 200001, size=n), # Random income amount
    "credit_score": np.random.randint(300, 851, size=n), # Random credit score
    "is_fraud": np.random.choice([0, 1], size=n, p=[.95, 0.05]) # Only 5% Fraud
}

# Make it into a Data Frame
users_df = pd.DataFrame(data)

In [4]:
# Each user makes 10 purchases (100,000 total)
n_trans = 100000
trans_data = {
    "trans_id": range(1, n_trans + 1),
    "user_id": np.random.choice(users_df["user_id"], size=n_trans),
    "amount": np.random.uniform(10.0, 500.0, size=n_trans),
    "category": np.random.choice(["Grocery", "Dining", "Electronics", "Travel"], size=n_trans),
    "hour": np.random.randint(0, 24, size=n_trans)
}

# Make it into a Data Frame
trans_df = pd.DataFrame(trans_data)

In [5]:
# Joining the data together
df = pd.merge(trans_df, users_df, on="user_id")

# Helpful variables to keep the code clean
fraud_mask = (df["is_fraud"] == 1)
num_fraudsters = fraud_mask.sum()

## Step 2: Implementing Fraudulent Behavior
In This section we will be implementing common fraudulent behavior within the data.

In [8]:
# IMPLEMENTING FRAUDULENT PATTERNS

# Pattern A: Frauds have stolen accounts with low credit scores and high incomes
df.loc[fraud_mask, "credit_score"] = np.random.randint(300, 501, size=num_fraudsters)
df.loc[fraud_mask, "income"] = np.random.randint(120001, 500001, size=num_fraudsters)

# Pattern B: Frauds often operate when most people are sleeping so they dont see bank alerts
df.loc[fraud_mask, "hour"] = np.random.randint(0, 5, size=num_fraudsters) # 12-5 AM

# Pattern C: Frauds disproportionately target high-risk, high-liquidity merchant categories
df.loc[fraud_mask, "category"] = np.random.choice(["Electronics", "Travel"], size=num_fraudsters)

## Step 3:
To evaluate detection performance under class imbalance (5% fraud distribution), we split the 100,000-transaction dataset into an 80/20 stratified train/test split. A baseline Random Forest Classifier (`n_estimators=100`, `max_depth=10`) was trained on 80,000 records and evaluated against 20,000 unseen held-out records.

In [7]:
# Cell [ ]: EVALUATION ON HELD-OUT TEST DATA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, precision_recall_fscore_support, roc_auc_score

# 1. One-hot encode categorical features ('category')
model_df = pd.get_dummies(df, columns=["category"], drop_first=True)

# 2. Select feature columns and target
features = ["amount", "hour", "income", "credit_score"] + [col for col in model_df.columns if col.startswith("category_")]
X = model_df[features]
y = model_df["is_fraud"]

# 3. 80/20 train/test split with stratification to preserve 5% fraud distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 4. Fit Random Forest baseline
clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

# 5. Evaluate on held-out test data
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="binary")
auc = roc_auc_score(y_test, y_prob)

print("--- HELD-OUT EVALUATION METRICS ---")
print(f"Held-out Test Rows: {len(y_test):,}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {auc:.4f}\n")
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"]))

--- HELD-OUT EVALUATION METRICS ---
Held-out Test Rows: 20,000
Precision: 1.0000
Recall:    0.7980
F1-Score:  0.8877
ROC-AUC:   0.9986

              precision    recall  f1-score   support

  Legitimate       0.99      1.00      1.00     19079
       Fraud       1.00      0.80      0.89       921

    accuracy                           0.99     20000
   macro avg       1.00      0.90      0.94     20000
weighted avg       0.99      0.99      0.99     20000



### Summary of Model Evaluation Findings
- **High-Fidelity Precision (1.0000):** The Random Forest classifier produced zero false positives across all 19,079 legitimate test transactions, ensuring genuine cardholder behavior is never misclassified.
- **Coverage & Recall (0.7980):** Caught 735 of the 921 fraudulent transactions (~80%) in the held-out test split, confirming strong signal extraction from the temporal, financial, and merchant-category feature patterns.
- **Overall Discriminative Power (0.9986 ROC-AUC):** Demonstrates near-optimal class separation between legitimate and fraudulent transactions under a realistic 5% class imbalance.